# 01 · Data quality & exploratory analysis

Audita la integridad del dataset antes de modelizar. Un modelo preciso sobre datos incoherentes no es útil.

In [1]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project8.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre el notebook dentro del repositorio del Proyecto 8.')
sys.path.insert(0,str(ROOT))
from src.project8 import *

df=pd.read_csv(ROOT/'data/raw/voyages_2026_synthetic.csv',parse_dates=['date'])

Validaciones: nulos, duplicados, capacidad, rangos meteorológicos y reconciliación económica.

In [2]:
qc={
'unique_voyage_id':df.voyage_id.is_unique,
'null_cells':int(df.isna().sum().sum()),
'pax_capacity_ok':bool((df.passengers<=df.pax_capacity).all()),
'fuel_positive':bool(df.fuel_tonnes.gt(0).all()),
'margin_reconciles':bool(np.allclose(df.margin_eur,df.revenue_eur-df.fuel_cost_eur-df.ets_cost_eur-df.variable_other_eur-df.fixed_trip_eur))
}
print(qc)
display(df[['fuel_tonnes','speed_kn','load_factor','sea_state','wind_kn','delay_min','margin_eur']].describe().T)

{'unique_voyage_id': True, 'null_cells': 0, 'pax_capacity_ok': True, 'fuel_positive': True, 'margin_reconciles': True}


,count,mean,std,min,25%,50%,75%,max
fuel_tonnes,7200.0,8.147959,3.061432,3.284446,5.402288,7.945039,10.470485,20.556991
speed_kn,7200.0,22.411696,3.141364,16.099486,20.111238,21.335847,25.138925,31.000000
load_factor,7200.0,0.728093,0.162537,0.350000,0.613688,0.732164,0.851554,0.990000
sea_state,7200.0,1.365694,1.137701,0.000000,1.000000,1.000000,2.000000,5.000000
wind_kn,7200.0,16.027820,6.851653,0.000000,11.234555,15.949979,20.755046,42.000000
delay_min,7200.0,6.570973,12.180066,0.000000,0.000000,0.000000,8.826567,99.097742
margin_eur,7200.0,41181.359500,18084.591836,4125.607224,27994.411930,39406.426656,52261.935695,120995.053676


Lectura agregada por ruta.

In [3]:
display(df.groupby(['route_id','route_name']).agg(voyages=('voyage_id','count'),fuel_t=('fuel_tonnes','sum'),otr15=('on_time15','mean'),margin=('margin_eur','sum')).round(3))

,,voyages,fuel_t,otr15,margin
route_id,route_name,,,,
BCN-IBZ,Barcelona–Ibiza,798,9833.889,0.709,4.049769e+07
BCN-PMI,Barcelona–Palma,863,7903.977,0.781,3.700504e+07
DEN-FOR,Dénia–Formentera,797,3818.900,0.960,2.720560e+07
DEN-IBZ,Dénia–Ibiza,1157,5714.174,0.939,4.094511e+07
DEN-PMI,Dénia–Palma,962,9849.907,0.760,4.367588e+07
PMI-IBZ,Palma–Ibiza,745,4220.519,0.933,2.759978e+07
VLC-IBZ,Valencia–Ibiza,980,7072.932,0.871,3.712967e+07
VLC-PMI,Valencia–Palma,898,10251.008,0.722,4.244703e+07
